# 05 - Dashboard Preparation

Train the final logistic regression models on all 2,411 genomes, save them, and
export the summary tables used by the dashboard.

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

ROOT = Path.home() / "amr-project"
PROC = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models"
DASH = ROOT / "dashboard" / "data"
MODEL_DIR.mkdir(exist_ok=True)
DASH.mkdir(parents=True, exist_ok=True)

X = pd.read_csv(PROC / "gene_matrix_filtered.csv", index_col=0, dtype={"genome_id": str})
y = pd.read_csv(PROC / "labels_clean.csv", index_col=0, dtype={"genome_id": str})
X.index = X.index.astype(str)
y.index = y.index.astype(str)
y = y.loc[X.index]
assert (X.index == y.index).all()
print("Features:", X.shape, " Labels:", y.shape)

Features: (2411, 102)  Labels: (2411, 3)


In [2]:
for drug in y.columns:
    lr = LogisticRegression(max_iter=2000, class_weight="balanced")
    lr.fit(X, y[drug].astype(int))
    joblib.dump({"model": lr, "genes": list(X.columns)},
                MODEL_DIR / f"{drug}_logreg.joblib")
    print("saved", drug)

saved ceftriaxone
saved ciprofloxacin
saved gentamicin


In [3]:
# 1. prevalence per drug
prev = pd.DataFrame({
    "drug": y.columns,
    "resistant": [int(y[d].sum()) for d in y.columns],
    "susceptible": [int((1 - y[d]).sum()) for d in y.columns],
})
prev["pct_resistant"] = (100 * prev["resistant"] / len(y)).round(1)
prev.to_csv(DASH / "prevalence.csv", index=False)

# 2. gene frequency in resistant vs susceptible genomes
rows = []
for d in y.columns:
    yy = y[d].astype(int)
    for g in X.columns:
        rows.append({"drug": d, "gene": g,
                     "pct_in_resistant": round(100 * X.loc[yy == 1, g].mean(), 1),
                     "pct_in_susceptible": round(100 * X.loc[yy == 0, g].mean(), 1)})
pd.DataFrame(rows).to_csv(DASH / "gene_frequency.csv", index=False)

# 3. model weights per gene and drug
coef_rows = []
for d in y.columns:
    bundle = joblib.load(MODEL_DIR / f"{d}_logreg.joblib")
    for g, c in zip(bundle["genes"], bundle["model"].coef_[0]):
        coef_rows.append({"drug": d, "gene": g, "weight": round(float(c), 3)})
pd.DataFrame(coef_rows).to_csv(DASH / "gene_weights.csv", index=False)

# 4. copy the existing results tables
for name in ["gene_associations.csv", "model_results.csv", "model_results_grouped.csv"]:
    pd.read_csv(PROC / name).to_csv(DASH / name, index=False)

print(sorted(p.name for p in DASH.glob("*.csv")))

['gene_associations.csv', 'gene_frequency.csv', 'gene_weights.csv', 'model_results.csv', 'model_results_grouped.csv', 'prevalence.csv']


In [4]:
def predict(genes_present):
    v = pd.DataFrame(0, index=[0], columns=X.columns)
    for g in genes_present:
        v.loc[0, g] = 1
    out = {}
    for d in y.columns:
        bundle = joblib.load(MODEL_DIR / f"{d}_logreg.joblib")
        out[d] = round(float(bundle["model"].predict_proba(v[bundle["genes"]])[0, 1]), 3)
    return out

print("No resistance genes:     ", predict([]))
print("CTX-M-15 + gyrA_S83I + parC_S80I:",
      predict(["blaCTX-M-15", "gyrA_S83I", "parC_S80I"]))

No resistance genes:      {'ceftriaxone': 0.02, 'ciprofloxacin': 0.008, 'gentamicin': 0.028}
CTX-M-15 + gyrA_S83I + parC_S80I: {'ceftriaxone': 0.799, 'ciprofloxacin': 0.615, 'gentamicin': 0.051}


In [5]:
g3 = ["blaCTX-M-15", "gyrA_S83I", "parC_S80I"]
mask = (X[g3] == 1).all(axis=1)
print("Genomes carrying all three:", int(mask.sum()))
print("Observed % resistant among them:")
print((y[mask].mean() * 100).round(1))

Genomes carrying all three: 788
Observed % resistant among them:
ceftriaxone      99.9
ciprofloxacin    99.9
gentamicin       68.9
dtype: float64


In [6]:
for d in y.columns:
    b = joblib.load(MODEL_DIR / f"{d}_logreg.joblib")
    p = b["model"].predict_proba(X[b["genes"]])[:, 1]
    print(f"{d}: mean score among the 788 = {p[mask].mean():.3f} | observed resistant = {y.loc[mask, d].mean():.3f}")

ceftriaxone: mean score among the 788 = 0.996 | observed resistant = 0.999
ciprofloxacin: mean score among the 788 = 0.995 | observed resistant = 0.999
gentamicin: mean score among the 788 = 0.705 | observed resistant = 0.689


In [7]:
X.to_csv(DASH / "gene_matrix_filtered.csv")
y.to_csv(DASH / "labels_clean.csv")
print(sorted(p.name for p in DASH.glob("*.csv")))

['gene_associations.csv', 'gene_frequency.csv', 'gene_matrix_filtered.csv', 'gene_weights.csv', 'labels_clean.csv', 'model_results.csv', 'model_results_grouped.csv', 'prevalence.csv']


## Summary

- Final logistic regression models (one per drug) were trained on all 2,411 genomes
  and 102 filtered genes, and saved to `models/` with `joblib`.
- Logistic regression was chosen because it performed as well as random forest and
  gradient boosting in cross-validation and its weights are easy to interpret.
- Eight tables were exported to `dashboard/data/`: prevalence, gene frequency by
  phenotype, model weights, gene associations, both model-results tables, and the
  filtered gene matrix with labels (so the dashboard can load real genomes).
- Outputs are model scores (class-weighted), not calibrated probabilities. A
  hand-built input of three genes (blaCTX-M-15, gyrA_S83I, parC_S80I) scores 0.80,
  0.62 and 0.05, but real genomes carrying those three score 0.996, 0.995 and 0.705
  on average (observed resistance 99.9%, 99.9%, 68.9%), because real genomes carry
  many co-occurring genes. These scores are in-sample and only confirm the scoring;
  accuracy was measured by cross-validation in Notebook 04.